# Private run #3 - Qwen2.5-Coder-14B N=5 consensus

Attach dataset `kien2005/kaggle-payload-private-v29-qwen14b-n5`, enable **GPU T4 x2** and Internet. The runner targets exactly 327 V29 single-vote rows and emits raw challengers for local verification.

In [ ]:
import glob, json, pathlib
EXPLICIT = pathlib.Path('/kaggle/input/datasets/kien2005/kaggle-payload-private-v29-qwen14b-n5')
hits = sorted(EXPLICIT.glob('**/retrieval.jsonl')) if EXPLICIT.exists() else []
if not hits:
    hits = [pathlib.Path(p) for p in glob.glob('/kaggle/input/**/retrieval.jsonl', recursive=True) if 'private-v29-qwen14b-n5' in p]
assert len(hits) == 1, f'Attach exactly one V29 private payload: {hits}'
PAYLOAD = str(hits[0].parent)
manifest = json.loads((pathlib.Path(PAYLOAD) / 'payload-manifest.json').read_text())
targets = json.loads((pathlib.Path(PAYLOAD) / 'target_ids.txt').read_text())['ids']
assert manifest['schema_version'] == 2 and len(targets) == 327
print('PAYLOAD', PAYLOAD, '| targets', len(targets), '| files', len(manifest['files']))
import torch
print('GPUs', torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
import pathlib, shutil
SRC, DST = pathlib.Path(PAYLOAD) / 'code', pathlib.Path('/kaggle/working/code')
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print('verified payload code copied to', DST)

In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes
print('transformers', transformers.__version__, 'bitsandbytes', bitsandbytes.__version__)

In [ ]:
%%time
# Full N=5 run. Re-running this cell resumes only with the same signature.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select --llm-target all --no-rule-fallback \
    --llm-ids-file $PAYLOAD/target_ids.txt \
    --out /kaggle/working/private_v29_qwen14b_n5_raw.jsonl \
    --n 5 --temperature 0.45 --k 15 --max-tokens 128 --batch-size 2 \
    --max-input-tokens 6000 --checkpoint-every 16 --time-budget-min 500 --seed 29

In [ ]:
import collections, json, math, pathlib
out = pathlib.Path('/kaggle/working/private_v29_qwen14b_n5_raw.jsonl')
rows = [json.loads(line) for line in out.open(encoding='utf-8')]
assert len(rows) == 1012 and len({r['id'] for r in rows}) == 1012
llm = [r for r in rows if str(r.get('source', '')).startswith('llm_select')]
assert {r['id'] for r in llm} <= set(targets)
assert all(math.isfinite(float(r['answer'])) for r in rows)
print('sources', collections.Counter(r['source'] for r in rows))
print('LLM votes', collections.Counter(r.get('votes', 0) for r in llm))
print('strict 4/5 candidates', sum(int(r.get('votes', 0)) >= 4 for r in llm))
print('DOWNLOAD', out)

Download `private_v29_qwen14b_n5_raw.jsonl`. Do not submit it directly. Local script `scripts/16_merge_consensus_codegen.py` verifies and merges it into V29.